# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type: Ranking / Scoring**, with a binary classification sub-component underneath.

The actual deliverable from Week 1's framing is a ranked review queue, not a single yes/no answer for one page — "which pages should an editor look at first" is a "which ones first?" question, and per the task-type mapping (ranking/scoring → priority score → precision@K), that's a ranking/scoring problem, not a plain classification one.

Classification still shows up as an ingredient: the starter pipeline predicts a binary decline probability (`is_declining_label`) and blends it with a rule-based score to produce the final ranking. So the model underneath is a classifier, but the *task* — the thing the editor actually receives and the thing this lane is graded on — is an ordered list with reason codes, which is scoring/ranking.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Working target for now:** the starter pipeline's `is_declining_label = (trend_direction == "down")`.

Being honest about what this is: it's a **defined-rule proxy, not a pure observed outcome**. `trend_direction` is computed by comparing the last-30-day window to the previous-30-day window *inside the same snapshot* — it describes where a page already is, not what happens to it next. That's a real limitation, not a technicality: Section 1 of the framing skill is explicit that "a label that comes from someone's rule means your model learns the rule, not the world," and this label sits closer to a rule than to a future-observed event.

I'm using it anyway for this week's framing exercise because it's the only label the starter data can currently support without warehouse access (unlocked Week 3) — but flagging it now, on purpose, so it doesn't quietly become "the truth" later. The lane guide's own recommendation for a stronger capstone target is a genuine future-window outcome:

```
features from prior 90 days -> decline or recovery over next 30 days
```

That's the target I plan to move to once I can build proper time-window features from `fact_content_daily_performance`.

**One more nuance for the ranking task specifically:** the *real* output isn't the binary label at all — it's the continuous `final_refresh_score` used to order pages. The binary label is one input signal that feeds that score, not the end product itself.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Precision@50.**

Week 1 named the actual decision constraint: an editor has limited weekly review capacity, not infinite time to sift 30,000 pages. The lane guide's own guidance for ranked-action work is explicit that top-K metrics beat generic accuracy because they match how the list is actually used — "Precision@50 if the team can act on 50 candidates" — and 50 is also the exact K the starter pipeline's own verified results are reported at, so it's directly comparable: baseline 0.240 vs. random forest 0.740 (`outputs/model_report.md`).

Why not plain accuracy: Week 1's numbers showed a 54.2% "down" base rate in this slice — high enough that a naive "always predict down" rule already looks decent on accuracy alone, which would make accuracy a misleading yardstick here.

**Secondary, not primary:** Average Precision, to sanity-check the ranking quality beyond just the top 50 (a model that's only good at the very top but random after that is a narrower win than one with a well-ordered full list). But Precision@50 is the one number I'd defend if asked "is this good?", because it's the one tied directly to the real decision.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page), summarized over its trailing 90-day window** — the same grain as `content_refresh_anonymized.csv` itself (`content_id` is unique per row).

Below: the lane-relevant slice of columns (identity, demand, position/CTR, freshness, and the current proxy label), plus a sketch of what the target column looks like today — a 0/1 flag with its real class balance, so the shape of the labeling problem is visible before any modeling starts.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# The lane-relevant slice: identity + the signals this lane's baseline/model actually use.
# trend_direction/trend_pct are included ONLY to build the sketch target below --
# per Week 1's leakage rule, they are never allowed into the feature set.
lane_cols = [
    "content_id", "client_id",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "freshness_tier",
    "word_count", "content_type",
    "trend_direction", "trend_pct",
]
lane_df = df[lane_cols].copy()

print(f"Unit of analysis: one row = one content item. Shape: {lane_df.shape}")
display(lane_df.head(5))

# Sketch of the target column, as it exists TODAY (the provisional proxy from Section 2)
lane_df["is_declining_label"] = (lane_df["trend_direction"] == "down").astype(int)
print("\nTarget sketch -- is_declining_label class balance:")
print(lane_df["is_declining_label"].value_counts(normalize=True).rename("share").round(3))
display(lane_df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]].head(5))

Unit of analysis: one row = one content item. Shape: (30000, 14)


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,freshness_tier,word_count,content_type,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,29,17,10.6,0.76,187,20,0-30,3221.0,keyword article,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,20.3,0.05,445,25,0-30,2481.0,keyword article,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,36.5,0.09,141,20,0-30,3515.0,keyword article,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,58,78,6.2,0.49,463,22,0-30,NaN,keyword article,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,44.0,0.13,263,14,0-30,2803.0,keyword article,down,-34.7



Target sketch -- is_declining_label class balance:
is_declining_label
1    0.542
0    0.458
Name: share, dtype: float64


,content_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1


## 5. Why ML beats a fixed rule here

The lane guide's starter pipeline already tried the fixed-rule version first — six hand-written reason codes (`stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`, `low_engagement_visible_page`), each a simple threshold on one or two columns. That's a fair baseline, and it's not useless: `outputs/model_report.md` shows it scores ROC AUC 0.627, better than a coin flip.

But it's measurably weak compared to a learned model on the same data:

| Method | ROC AUC | Precision@50 |
|---|---:|---:|
| baseline rules | 0.627 | 0.240 |
| random forest | 0.750 | 0.740 |

That's roughly 12 correct pages in the baseline's top 50 versus 37 in the random forest's top 50 — a real, measured gap, not a hunch.

**Why the gap exists:** each rule only looks at one or two columns crossing one threshold. A page that's *moderately* declining, *moderately* under-performing on CTR, and *moderately* stale — none of it individually crossing any single rule's cutoff — is invisible to every rule at once, even though the combination is real signal. A model can weigh partial evidence across many interacting columns (trend, position, CTR, freshness tier, word-count tier, content type, intent) at the same time; a hand-written if-statement can't do that without turning into dozens of brittle nested conditions nobody could maintain. That's exactly the "many signals, tangled, shifting over time" condition where ML earns its place instead of a dashboard.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.